# 🇮🇳 India Finance RAG — Master Notebook

## Run Order Every Session
| Cell | What | Every session? |
|------|------|----------------|
| 1 | Mount Drive + Pull GitHub | ✅ Yes |
| 2 | Install packages | ✅ Yes |
| 3 | Load API keys | ✅ Yes |
| 4 | Health check | First time only |
| 5 | Run ingestion (Week 2+3) | First time only |
| 6 | Test retrieval (Week 4) | After ingestion |
| 7 | Test agents (Week 5+6) | After Week 4 |
| 8 | Run Streamlit UI (Week 7) | When demoing |
| 9 | Run evaluation (Week 8) | Before submission |

In [ ]:
# ============================================================
# CELL 1 — ALWAYS RUN FIRST
# Mount Google Drive + Pull latest code from GitHub
# ============================================================

from google.colab import drive
import os

# Mount Drive — click the link and sign in
drive.mount('/content/drive')

# Create project folders on Drive
for folder in [
    '/content/drive/MyDrive/Finance_RAG',
    '/content/drive/MyDrive/Finance_RAG/finrag_db',
    '/content/drive/MyDrive/Finance_RAG/logs',
]:
    os.makedirs(folder, exist_ok=True)

# Clone or pull your GitHub repo
# ⚠️ CHANGE THIS to your actual GitHub repo URL
REPO_URL  = 'https://github.com/arya-soundu/india-finance-rag.git'
REPO_NAME = 'india-finance-rag'

if os.path.exists(f'/content/{REPO_NAME}'):
    print('Repo exists — pulling latest changes...')
    os.chdir(f'/content/{REPO_NAME}')
    os.system('git pull origin main')
else:
    print('Cloning repo for the first time...')
    os.chdir('/content')
    os.system(f'git clone {REPO_URL}')
    os.chdir(f'/content/{REPO_NAME}')

import sys
sys.path.insert(0, f'/content/{REPO_NAME}')

print(f'\n✅ Working in: {os.getcwd()}')
print(f'Files: {os.listdir(".")}')

In [ ]:
# ============================================================
# CELL 2 — ALWAYS RUN (takes 2-3 mins)
# Install all required packages
# ============================================================

%pip install -q -r requirements.txt
print('✅ All packages installed!')

In [ ]:
# ============================================================
# CELL 3 — ALWAYS RUN
# Load API keys from Colab Secrets
# ============================================================
# BEFORE running this cell:
#   1. Click the 🔑 key icon in the left sidebar
#   2. Add: GROQ_API_KEY, LANGCHAIN_API_KEY, TAVILY_API_KEY
#   3. Toggle 'Notebook access' ON for each key
# ============================================================

from google.colab import userdata
import os

os.environ['GROQ_API_KEY']           = userdata.get('GROQ_API_KEY')
os.environ['LANGCHAIN_API_KEY']      = userdata.get('LANGCHAIN_API_KEY')
os.environ['LANGCHAIN_TRACING_V2']   = 'true'
os.environ['LANGCHAIN_PROJECT']      = 'india-finance-rag'

try:
    os.environ['TAVILY_API_KEY'] = userdata.get('TAVILY_API_KEY')
    print('✅ All keys loaded (Groq + LangSmith + Tavily)')
except Exception:
    print('✅ Core keys loaded (Tavily optional — needed for Web Search Agent)')

In [ ]:
# ============================================================
# CELL 4 — RUN ONCE to confirm everything works
# Health check
# ============================================================

print('Testing all components...\n')

# Test Groq LLM
from langchain_groq import ChatGroq
llm = ChatGroq(model='llama3-8b-8192', temperature=0, api_key=os.environ['GROQ_API_KEY'])
r = llm.invoke('What is SEBI? One sentence only.')
print(f'✅ Groq LLM: {r.content[:80]}...')

# Test ChromaDB
import chromadb
print('✅ ChromaDB ready')

# Test yFinance with Indian stock
import yfinance as yf
tcs = yf.Ticker('TCS.NS')
price = tcs.info.get('currentPrice', 'N/A')
print(f'✅ yFinance: TCS.NS current price = ₹{price}')

# Test web scraping
import requests
from bs4 import BeautifulSoup
print('✅ requests + BeautifulSoup ready')

# Test PDF
import pdfplumber
print('✅ pdfplumber ready')

# Test sentence transformers
from sentence_transformers import SentenceTransformer
print('✅ SentenceTransformer ready (will download ~90MB on first encode)')

print('\n🚀 All systems go!')

In [ ]:
# ============================================================
# CELL 5 — WEEK 2 + 3: Run Ingestion Pipeline
# RUN ONCE (or when adding new documents)
# Expected time: 10-20 minutes
# ============================================================

print('Starting ingestion pipeline...')
print('This fetches Indian company data, SEBI regulations, and BSE reports.')
print('Expected time: 10-20 minutes\n')

from src.ingest import main as run_ingestion
run_ingestion()

print('\n✅ Ingestion complete!')

In [ ]:
# ============================================================
# CELL 6 — WEEK 4: Test Retrieval
# Confirms ChromaDB search is working
# ============================================================

from src.retriever import get_retriever

# Load retriever (connects to ChromaDB on Drive)
retriever = get_retriever()

# Show DB statistics
stats = retriever.get_db_stats()
print(f'Total chunks in database: {stats["total_chunks"]}')
print(f'Companies covered: {stats["companies"]}')
print(f'Document types: {stats["doc_types"]}')

# Test search
print('\nTest searches:')
test_queries = [
    'What is SEBI LODR regulation?',
    'What is TCS revenue?',
    'What is insider trading?',
]

for q in test_queries:
    results = retriever.search(q, n_results=2)
    print(f'\nQuery: {q}')
    for r in results:
        print(f'  Score: {r["score"]:.3f} | {r["company"]} | {r["filing_type"]}')
        print(f'  Text: {r["text"][:100]}...')

In [ ]:
# ============================================================
# CELL 7 — WEEK 5 + 6: Test Agents
# Tests all three agents with sample queries
# ============================================================

from src.agents import process_query
from src.retriever import get_retriever

retriever = get_retriever()

test_cases = [
    # Tests RAG agent
    ('What are SEBI insider trading regulations?', 'RAG'),
    # Tests Calculator agent
    ('Calculate P/E ratio if EPS is 85 and share price is 3400', 'Calculator'),
    # Tests Web Search agent
    ('What is the latest news about Reliance Industries today?', 'Web Search'),
]

for question, expected_agent in test_cases:
    print(f'\n{"="*60}')
    print(f'Question: {question}')
    print(f'Expected agent: {expected_agent}')
    
    result = process_query(question, retriever)
    
    print(f'Agent used:   {result["agent_used"]}')
    print(f'Confidence:   {result["confidence"]}')
    print(f'Sources:      {len(result["sources"])} chunks')
    print(f'Answer:       {result["answer"][:300]}...')

In [ ]:
# ============================================================
# CELL 8 — WEEK 7: Run Streamlit UI in Colab
# Uses pyngrok to create a public URL
# ============================================================

# Install ngrok tunnel (needed to expose Streamlit in Colab)
%pip install -q pyngrok

from pyngrok import ngrok
import subprocess
import time

# Start Streamlit in background
streamlit_proc = subprocess.Popen(
    ['streamlit', 'run', 'src/app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Wait for Streamlit to start
time.sleep(5)

# Create public tunnel
public_url = ngrok.connect(8501)
print(f'\n✅ Streamlit is running!')
print(f'\n👉 Open this URL in your browser:')
print(f'   {public_url}')
print(f'\n(Keep this cell running to keep the app alive)')
print(f'(Stop the cell to shut down the app)')

# Keep running until manually stopped
try:
    streamlit_proc.wait()
except KeyboardInterrupt:
    streamlit_proc.terminate()
    ngrok.kill()
    print('App stopped.')

In [ ]:
# ============================================================
# CELL 9 — WEEK 8: Run RAGAS Evaluation
# Tests 30 questions and produces evaluation report
# Expected time: 15-25 minutes
# ============================================================

from src.evaluate import run_evaluation

print('Starting RAGAS-style evaluation...')
print('Testing 30 questions across all document categories')
print('Expected time: 15-25 minutes\n')

summary = run_evaluation()

# Print final scores
print('\n' + '='*60)
print('FINAL EVALUATION SCORES')
print('='*60)
for metric, score in summary['metrics'].items():
    grade = '🟢 Excellent' if score >= 0.8 else ('🟡 Good' if score >= 0.6 else '🔴 Needs work')
    print(f'{metric:35s}: {score:.3f}  {grade}')

print('\nPer-category breakdown:')
for cat, stats in summary['per_category'].items():
    print(f'  {cat:25s}: overall={stats["avg_overall"]:.3f} (n={stats["count"]})')

report_path = '/content/drive/MyDrive/Finance_RAG/logs/ragas_evaluation_report.csv'
print(f'\n✅ Full CSV report saved to: {report_path}')

In [ ]:
# ============================================================
# CELL 10 — Interactive Chat (no UI, just terminal)
# For quick testing without launching Streamlit
# ============================================================

from src.agents import process_query
from src.retriever import get_retriever

retriever = get_retriever()

print('India Finance RAG — Terminal Chat Mode')
print('Type your question and press Enter.')
print('Type "quit" to exit.\n')

while True:
    try:
        query = input('You: ').strip()
        if not query:
            continue
        if query.lower() in ['quit', 'exit', 'q']:
            print('Goodbye!')
            break
        
        result = process_query(query, retriever)
        
        print(f'\nAgent: {result["agent_used"]} | Confidence: {result["confidence"]:.2f}')
        print(f'Answer: {result["answer"]}')
        
        if result['sources']:
            print(f'\nSources ({len(result["sources"])}):')
            for s in result['sources'][:2]:
                print(f'  • {s["company"]} | {s["filing_type"]} | score: {s["score"]:.2f}')
        print()
        
    except KeyboardInterrupt:
        print('\nChat ended.')
        break